<a href="https://colab.research.google.com/github/ErickMBarreto/HackatonFiap/blob/main/AiSecurityEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
# -----------------------------------------------------------
# @title 01 - Inicialização do Núcleo
# AI SECURITY ENGINE
# -----------------------------------------------------------

import os
import sys
import requests

DATASET_URL = 'https://raw.githubusercontent.com/ErickMBarreto/HackatonFiap/refs/heads/main/dataset/dataset_treino.json'
DATASET_LOCAL = 'dataset_treino.json'

def download_dataset_if_missing():
    if not os.path.exists(DATASET_LOCAL):
        print(f"🔄 Dataset não encontrado localmente. Baixando de {DATASET_URL}...")
        try:
            response = requests.get(DATASET_URL)
            if response.status_code == 200:
                with open(DATASET_LOCAL, 'wb') as f:
                    f.write(response.content)
                print("📱 Dataset baixado com sucesso!")
            else:
                print(f"📲 Erro ao baixar dataset (Status: {response.status_code}). Verifique a URL.")
                print("🔂 Ação manual necessária: Faça o upload do arquivo 'dataset_treino.json'.")
        except Exception as e:
            print(f"📲 Falha na conexão: {e}")
    else:
        print("📱 Dataset local detectado.")

# Executa a verificação
download_dataset_if_missing()


# Instalação silenciosa e robusta
if 'fpdf' not in sys.modules:
    os.system('pip install -q fpdf')

import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage
from fpdf import FPDF
import json
import time
import io
import random
from datetime import datetime

# --- CONFIGURAÇÃO ---
print("🔋 Inicializando Sistema...")

try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
except Exception:
    API_KEY = input("🔑 Insira sua API KEY: ").strip()
    genai.configure(api_key=API_KEY)

MODEL_NAME = "gemini-2.5-flash"
DATASET_PATH = 'dataset_treino.json'

KNOWLEDGE_BASE = []
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, 'r') as f:
        KNOWLEDGE_BASE = json.load(f)
    print("📱 Base carregada")

# --- GERADOR DE PDF ---

class PDFReport(FPDF):
    def header(self):
        self.set_fill_color(33, 37, 41)
        self.rect(0, 0, 210, 35, 'F')
        self.set_font('Helvetica', 'B', 18); self.set_text_color(255, 255, 255)
        self.set_xy(10, 10); self.cell(0, 10, 'AI SECURITY ENGINE', ln=True)
        self.set_font('Helvetica', 'I', 9); self.set_xy(10, 18)
        self.cell(0, 10, f'Relatorio Estrategico STRIDE - {datetime.now().strftime("%d/%m/%Y")}', ln=True)
        self.ln(18)

    def footer(self):
        self.set_y(-15); self.set_font('Helvetica', 'I', 8); self.set_text_color(128, 128, 128)
        self.cell(0, 10, f'Pagina {self.page_no()} | AI Security Engine', align='C')

    def chapter_title(self, title):
        self.set_font('Helvetica', 'B', 12); self.set_text_color(33, 37, 41)
        self.set_fill_color(245, 245, 245); self.cell(0, 10, f" {title}", ln=True, fill=True); self.ln(3)

    def chapter_body(self, body):
        self.set_font('Helvetica', '', 10); self.set_text_color(60, 60, 60)
        txt = body.encode('latin-1', 'replace').decode('latin-1')
        self.multi_cell(0, 6, txt); self.ln(2)

    def add_threat_card(self, threat):
        def clean(t):
            return str(t).encode('latin-1', 'replace').decode('latin-1')

        # 1. Coletar e limpar dados
        category = clean(threat.get('category') or "N/A")
        component = clean(threat.get('component', 'N/A'))
        description = clean(threat.get('description', ''))
        mitigation = clean(threat.get('mitigation', ''))

        # 2. Verificar quebra de página (Aumentei a margem para 220mm por segurança)
        if self.get_y() > 220:
            self.add_page()

        # 3. Guardar posição inicial para o retângulo
        y_inicial = self.get_y()

        # --- CONTEÚDO DO CARD ---
        # Título Categoria
        self.set_xy(13, y_inicial + 3)
        self.set_font('Helvetica', 'B', 10)
        self.set_text_color(180, 0, 0)
        self.cell(0, 6, f"CATEGORIA: {category.upper()}", ln=True)

        # Componente Alvo (Agora com multi_cell para nomes grandes)
        self.set_x(13)
        self.set_font('Helvetica', 'B', 9)
        self.set_text_color(33, 37, 41)
        self.multi_cell(180, 5, f"COMPONENTE ALVO: {component}")

        # Descrição
        self.set_x(13)
        self.set_font('Helvetica', '', 9)
        self.set_text_color(60, 60, 60)
        self.multi_cell(180, 4.5, f"Descricao: {description}")

        # Mitigação Sugerida
        self.set_x(13)
        self.set_font('Helvetica', 'B', 9)
        self.set_text_color(40, 120, 40)
        self.multi_cell(180, 5, f"MITIGACAO SUGERIDA: {mitigation}")

        # 4. DESENHO DO RETÂNGULO DINÂMICO
        # Em vez de altura fixa (38), calculamos a diferença entre o fim do texto e o início
        y_final = self.get_y()
        altura_card = y_final - y_inicial + 2 # +2 para dar um respiro no fundo

        self.set_draw_color(220, 220, 220)
        self.rect(10, y_inicial, 190, altura_card)

        # Ajusta o cursor para o próximo card com um espaçamento fixo
        self.set_y(y_final + 8)

# --- RE-EXECUÇÃO DA INTERFACE (Mantenha o resto do código do motor de IA igual) ---
# --- MOTOR DE IA (In-Context) ---
def analyze_with_ai(image_path):
    # Prompt com injeção de exemplos (Few-Shot)
    examples = [ex for ex in KNOWLEDGE_BASE if ex.get('image_filename') != image_path]
    selected = random.sample(examples, min(2, len(examples))) if examples else []

    prompt = ["VOCÊ É UM ESPECIALISTA EM APPSEC (STRIDE). GERE APENAS JSON."]
    prompt.append("Estrutura Obrigatória: { 'assets': [], 'threats': [{ 'component', 'category', 'description', 'mitigation' }] }")
    prompt.append("Traduza a descrição e mitigação para Português do Brasil (PT-BR) formal.")

    for ex in selected:
        if os.path.exists(ex.get('image_filename', '')):
            prompt.append(f"Exemplo de Treino (Input): Analise este diagrama.")
            prompt.append(f"Exemplo de Treino (Output Ideal): {json.dumps(ex['ground_truth_json'])}")

    prompt.append("AGORA, ANALISE A IMAGEM ABAIXO:")

    myfile = genai.upload_file(image_path)
    while myfile.state.name == "PROCESSING":
        time.sleep(1)
        myfile = genai.get_file(myfile.name)

    prompt.append(myfile)

    model = genai.GenerativeModel(MODEL_NAME)
    res = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
    return res.text



📱 Dataset local detectado.
🔋 Inicializando Sistema...
📱 Base carregada


In [54]:
# @title 2. INTERFACE [AI Security Engine]

import base64
import re
import json
import time
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


def clean_json_markdown(text):
    if not text: return "{}"
    text = re.sub(r'```json\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'```', '', text)
    return text.strip()

def create_base64_link(filename, title="Download PDF"):
    try:
        with open(filename, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()

        html = f"""
            <div style="margin-top:15px; text-align: center;">
                <a href="data:application/pdf;base64,{b64}" download="{filename}" target="_blank"
                   style="background-color:#333; color:#FFF; padding:10px 24px;
                          text-decoration:none; font-weight:600; border-radius:8px;
                          font-family:sans-serif; border: 1px solid #000;
                          display:inline-block;">
                    📥 {title}
                </a>
            </div>
        """
        return html
    except Exception as e:
        return f"<div style='color:red'>Erro ao gerar link: {str(e)}</div>"

# --- ESTILO VISUAL ---
css = """
<style>
    .panel {
        background-color: #FFFFFF;
        padding: 30px 25px 20px 25px;
        border: 1px solid #E0E0E0;
        border-radius: 16px;
        max-width: 700px;
        margin: 0 auto;
        box-shadow: 0 10px 30px rgba(0,0,0,0.05);
        font-family: 'Segoe UI', system-ui, sans-serif;
    }
    .title {
        color: #111111 !important;
        font-weight: 800;
        font-size: 28px;
        letter-spacing: -1px;
        text-align: center;
        display: block;
    }
    .subtitle {
        color: #888888;
        font-size: 14px;
        text-align: center;
        display: block;
        margin-bottom: 25px;
    }
    .log {
        background-color: #FBFBFB;
        color: #444444;
        font-size: 13px;
        line-height: 1.6;
        border-radius: 8px;
    }
    .log-active {
        border: 1px solid #F0F0F0;
        padding: 15px;
        margin-top: 15px;
    }
    .engine-btn {
        background-color: #333333 !important;
        color: #FFFFFF !important;
        font-weight: 600 !important;
        border-radius: 8px !important;
    }
    .center-content {
        display: flex;
        flex-direction: column;
        align-items: center;
        width: 100%;
    }
</style>
"""
display(HTML(css))

# --- WIDGETS ---
header = widgets.HTML("""
    <span class='title'>AI Security Engine</span>
    <span class='subtitle'>Threat Modeling & Intelligent Analysis</span>
""")

btn_upload = widgets.FileUpload(accept='image/*', multiple=False, description='Selecionar Diagrama', layout=widgets.Layout(width='220px'))
btn_run = widgets.Button(description='INICIAR ANÁLISE ESTRATÉGICA', layout=widgets.Layout(width='100%', height='45px', margin='10px 0'))
btn_run.add_class('btn')

output_log = widgets.Output()
output_log.add_class('log')
output_file = widgets.HTML()

ui_elements = widgets.VBox([header, btn_upload, btn_run, output_log, output_file])
ui_elements.add_class('center-content')
ui = widgets.VBox([ui_elements])
ui.add_class('panel')

# --- LÓGICA DE EXECUÇÃO (RESTAURADA INTEGRALMENTE) ---

def on_upload_change(change):
    if not change['new']: return
    output_log.clear_output()
    output_log.add_class('log-active')
    output_file.value = ""
    with output_log:
        print(f"> [IO] ARQUIVO DETECTADO: {list(change['new'].keys())[0]}")
        uploaded_file = list(change['new'].values())[0]
        display(HTML("<div style='text-align:center; margin-top:10px;'>"))
        display(widgets.Image(value=uploaded_file['content'], width=400))
        display(HTML("</div>"))
        print("> IMAGEM PRONTA PARA ANÁLISE.")

btn_upload.observe(on_upload_change, names='value')

def on_run_click(b):
    output_log.clear_output()
    output_log.add_class('log-active')
    output_file.value = ""

    with output_log:
        print("> INICIANDO PROTOCOLO...")

        if not btn_upload.value:
            print(f"> [ERRO] Nenhuma imagem carregada.")
            return

        try:
            print("> LENDO ARQUIVO DE IMAGEM...")
            uploaded_file = list(btn_upload.value.values())[0]
            filename = "temp_diagram.png"
            with open(filename, "wb") as f:
                f.write(uploaded_file['content'])

            print("> [AGUARDE] ENVIANDO PARA GEMINI 2.5 FLASH...")
            start_t = time.time()

            # CHAMADA DAS FUNÇÕES DA CÉLULA 02
            raw_json_str = analyze_with_ai(filename)

            elapsed = time.time() - start_t
            print(f"> RESPOSTA RECEBIDA EM {elapsed:.2f}s")

            print("> PROCESSANDO ESTRUTURA LÓGICA...")
            clean_str = clean_json_markdown(raw_json_str)
            data = json.loads(clean_str)

            print("> DIAGRAMANDO RELATÓRIO PDF...")
            pdf = PDFReport()
            pdf.add_page()

            clean_assets = [str(a).encode('latin-1', 'replace').decode('latin-1') for a in data.get('assets', [])]
            pdf.chapter_title(f"1. Ativos Identificados ({len(clean_assets)})")
            pdf.chapter_body(", ".join(clean_assets))

            threats_list = data.get('threats', [])
            pdf.chapter_title(f"2. Matriz de Ameaças ({len(threats_list)})")
            for threat in threats_list:
                pdf.add_threat_card(threat)

            final_pdf_name = "Relatorio_Final.pdf"
            pdf.output(final_pdf_name)
            print(f"> RELATÓRIO GERADO: {final_pdf_name}")

            link_html = create_base64_link(final_pdf_name)
            output_file.value = link_html
            print("> PRONTO.")

        except Exception as e:
            print(f"> [ERRO FATAL] {str(e)}")

btn_run.on_click(on_run_click)

display(ui)